# T2.2 — Semantic Mapping Upload to DBRepo

Uploads all semantic mappings from `docs/semantic_mapping.csv` to DBRepo via REST API.

**Owner:** Person B | **Task:** T2.2 — Semantic Mapping | **Dataset:** Hohe Warte Vienna Weather

> **Requires:** TU Wien VPN active, and the group database already created in DBRepo (T2.1 complete).

## Step 0 — Install the official DBRepo Python library

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'dbrepo', '--quiet'])
print('dbrepo library ready')

# Verify the exact method signature we will use
from dbrepo.RestClient import RestClient
import inspect
print()
print('update_table_column signature:')
print(inspect.signature(RestClient.update_table_column))
print(inspect.getdoc(RestClient.update_table_column))


## Step 1 — Configuration

> Password is loaded from the `DBREPO_PASSWORD` environment variable, or prompted at runtime.
> **Never hardcode or commit your real password.**

In [ ]:
import os

ENDPOINT    = "https://dbrepo1.ec.tuwien.ac.at"          # TU Wien DBRepo instance
DATABASE_ID = "899bfcba-7fec-40c9-9076-3a3a9372c844"     # group database UUID
USERNAME    = "azra1558"                                  # DBRepo username

PASSWORD = "Katalizator1558!"

MAPPING_FILE = "../docs/semantic_mapping.csv"

assert os.path.exists(MAPPING_FILE), (
    f"Mapping file not found at: {os.path.abspath(MAPPING_FILE)}\n"
    f"Make sure you are running this notebook from the 'notebooks/' directory."
)
print(f"Mapping file found: {os.path.abspath(MAPPING_FILE)}")


## Step 2 — Connect to DBRepo and fetch database structure

The API requires **table UUID** and **column UUID** — not names. We fetch them here.

In [ ]:
from dbrepo.RestClient import RestClient

# RestClient handles authentication internally
client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)

db = client.get_database(database_id=DATABASE_ID)
print(f"Connected to database: {db.name}")

# db.tables is Optional — guard against None
tables = db.tables or []
print(f"Tables found: {[t.name for t in tables]}")
if not tables:
    raise RuntimeError("No tables found — check DATABASE_ID or ensure T2.1 is complete")


## Step 3 — Build name-to-ID lookup maps

In [ ]:
table_id_map  = {}  # table_name -> table UUID
column_id_map = {}  # (table_name, column_name) -> column UUID

for table in tables:
    table_id_map[table.name] = table.id
    # get_database may return tables without column details (lightweight response)
    # fetch each table individually to guarantee columns are populated
    if not table.columns:
        table = client.get_table(database_id=DATABASE_ID, table_id=table.id)
    print(f"  Table '{table.name}' -> {table.id} ({len(table.columns)} columns)")
    for col in table.columns:
        column_id_map[(table.name, col.name)] = col.id

print(f"\nMapped {len(table_id_map)} tables, {len(column_id_map)} columns")
if len(column_id_map) == 0:
    raise RuntimeError("No columns found — cannot upload. Check DBRepo schema.")

# Sanity check — print all discovered (table, column) pairs (fix point 4)
print("\nDiscovered columns in DBRepo:")
for k in sorted(column_id_map.keys()):
    print(f"  {k[0]}.{k[1]}")


## Step 4 — Load the semantic mapping CSV

In [ ]:
import csv

mappings = []
with open(MAPPING_FILE, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        mappings.append(row)

print(f"Loaded {len(mappings)} mappings from CSV")
for m in mappings[:3]:
    print(m)


## Step 5 — Upload each semantic concept using the official dbrepo client

Uses `client.update_table_column(database_id, table_id, column_id, concept_uri=uri)`.

> **Note on `ontology_label`:** Only `concept_uri` is sent to DBRepo. The label field in the CSV is for human readability. DBRepo resolves the concept name internally from its own concept store using the URI — no separate label call is needed.

In [ ]:
success, skipped, errors = 0, 0, []

for row in mappings:
    tname = row["table_name"]
    cname = row["column_name"]
    uri   = row["ontology_uri"]
    # ontology_label is human-readable only — DBRepo derives label from URI via its concept store

    tid = table_id_map.get(tname)
    cid = column_id_map.get((tname, cname))

    if tid is None or cid is None:
        skipped += 1
        print(f"  SKIP  {tname}.{cname} — not found in DBRepo schema")
        continue

    try:
        # Verified signature: update_table_column(database_id, table_id, column_id,
        #                                         concept_uri=None, unit_uri=None)
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=tid,
            column_id=cid,
            concept_uri=uri
        )
        success += 1
        print(f"  OK    {tname}.{cname} -> {uri}")
    except Exception as e:
        errors.append((tname, cname, str(e)))
        print(f"  FAIL  {tname}.{cname} -> {e}")

print()
print(f"Result: {success} uploaded, {skipped} skipped, {len(errors)} failed")
if errors:
    print("\nFailed rows:")
    for e in errors:
        print(f"  {e[0]}.{e[1]}: {e[2]}")


## Step 6 — Verify: read back spot-checks from DBRepo

In [ ]:
spot_checks = [
    ("weather_measurement", "t_mean_c"),
    ("weather_measurement", "precp_sum_mm"),
    ("station",             "nuts_code"),
    ("station",             "latitude_deg"),
    ("time_dimension",      "ref_year"),
]

print("Spot-check verification:")
all_ok = True
for tname, cname in spot_checks:
    tid = table_id_map.get(tname)
    cid = column_id_map.get((tname, cname))
    if tid is None or cid is None:
        print(f"  SKIP  {tname}.{cname} — ID not found")
        continue
    try:
        tbl = client.get_table(database_id=DATABASE_ID, table_id=tid)  # returns Table object
        matched = next((c for c in tbl.columns if c.name == cname), None)
        if matched:
            print(f"  OK    {tname}.{cname}")
            print(f"        concept_uri = {matched.concept_uri}")
            if not matched.concept_uri:
                print(f"        WARNING: concept_uri is empty — upload may have failed")
                all_ok = False
        else:
            print(f"  MISS  {tname}.{cname} — column not in response")
            all_ok = False
    except Exception as e:
        print(f"  FAIL  {tname}.{cname} -> {e}")
        all_ok = False

print()
print("All spot-checks passed" if all_ok else "Some checks failed — review above")
